In [ ]:
# 1. Install Libraries for Audio, Video, and Text processing
!pip install moviepy librosa transformers torch torchvision opencv-python tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Check PyTorch version to install the correct binaries
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")

# Install PyTorch Geometric and dependencies using pre-built wheels (FAST)
# This prevents the "Building wheels... stuck" error
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-{torch.__version__}.html

!pip install torch_geometric
!pip install moviepy librosa transformers opencv-python tqdm

PyTorch Version: 2.9.0+cu126
CUDA Version: 12.6
Looking in links: https://data.pyg.org/whl/torch-2.9.0+cu126.html
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import cv2
import librosa
import moviepy.editor as mp
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/Honors/'
JSON_FILE = os.path.join(BASE_PATH, 'sarcasm_data.json')
UTTERANCE_DIR = os.path.join(BASE_PATH, 'Utterance Videos/')
CONTEXT_DIR = os.path.join(BASE_PATH, 'Context Videos/') # Assumed name

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- MODEL LOADING ---
print("Loading Models (RoBERTa & ResNet)...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
text_model = RobertaModel.from_pretrained('roberta-base').to(device)

resnet = models.resnet50(pretrained=True)
modules = list(resnet.children())[:-1]
resnet_feat = nn.Sequential(*modules).to(device)
resnet_feat.eval()

# Helper: Visual Preprocessing
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- EXTRACTOR FUNCTIONS ---
def extract_text(text):
    if not text: return np.zeros(768)
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = text_model(**inputs)
    return out.last_hidden_state[:, 0, :].cpu().numpy().flatten()

def extract_video(path):
    if not os.path.exists(path): return np.zeros(2048)
    cap = cv2.VideoCapture(path)
    frames = []
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if count % 10 == 0: # Check every 10th frame
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            input_tensor = preprocess(frame).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = resnet_feat(input_tensor)
                frames.append(feat.cpu().numpy().flatten())
        count += 1
    cap.release()
    return np.mean(frames, axis=0) if frames else np.zeros(2048)

def extract_audio(path):
    if not os.path.exists(path): return np.zeros(15)
    try:
        video = mp.VideoFileClip(path)
        if video.audio is None: return np.zeros(15)
        video.audio.write_audiofile("temp.wav", verbose=False, logger=None)
        y, sr = librosa.load("temp.wav", sr=16000)

        mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
        pitches, _ = librosa.piptrack(y=y, sr=sr)
        pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0
        cent_mean = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

        os.remove("temp.wav")
        return np.hstack([mfcc, pitch_mean, cent_mean])
    except:
        return np.zeros(15)

# --- MAIN LOOP ---
print("Starting Extraction...")
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

text_f, audio_f, video_f, context_f, labels = [], [], [], [], []

for key, entry in tqdm(data.items()):
    labels.append(1 if entry['sarcasm'] else 0)

    # 1. Utterance
    text_f.append(extract_text(entry['utterance']))

    vid_path = os.path.join(UTTERANCE_DIR, f"{key}.mp4")
    video_f.append(extract_video(vid_path))
    audio_f.append(extract_audio(vid_path))

    # 2. Context (Text + Visual)
    c_text_str = " ".join(entry['context']) if isinstance(entry['context'], list) else entry['context']
    c_text_vec = extract_text(c_text_str) # 768 dim

    # Try finding context video (assuming filename matches key)
    c_vid_path = os.path.join(CONTEXT_DIR, f"{key}.mp4")
    c_vis_vec = extract_video(c_vid_path) # 2048 dim

    # Concatenate Context
    context_f.append(np.hstack([c_text_vec, c_vis_vec]))

# Save
np.save(os.path.join(BASE_PATH, 'text_features.npy'), np.array(text_f))
np.save(os.path.join(BASE_PATH, 'audio_features.npy'), np.array(audio_f))
np.save(os.path.join(BASE_PATH, 'video_features.npy'), np.array(video_f))
np.save(os.path.join(BASE_PATH, 'context_features.npy'), np.array(context_f))
np.save(os.path.join(BASE_PATH, 'labels.npy'), np.array(labels))

print("Extraction Done! Proceed to Step 3.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Loading Models (RoBERTa & ResNet)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  warnings.warn(

  warnings.warn(msg)



Starting Extraction...


 41%|████      | 280/690 [07:28<10:11,  1.49s/it]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/moviepy/audio/io/readers.py:197: UserWarning: Error in file /content/drive/MyDrive/Honors/Utterance Videos/2_223.mp4, At time t=11.49-11.54 seconds, indices wanted: 99343-101338, but len(buffer)=99344
index 99344 is out of bounds for axis 0 with size 99344
  warnings.warn("Error in file %s, "%(self.filename)+

100%|██████████| 690/690 [19:03<00:00,  1.66s/it]

Extraction Done! Proceed to Step 3.
